# 18 Integrated Gradients XAI sensitivity analysis

Generates Integrated Gradients attribution maps using the locked ResNet18 checkpoint and the same locked test set used for Grad-CAM. This provides an additional XAI method without requiring Captum.

In [7]:

from pathlib import Path
import os, json, glob, shutil, warnings, math, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd

SEED = int(os.environ.get("THERMO_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)

_candidate_bases = [Path(os.environ.get("THERMO_BASE_DIR", "")) if os.environ.get("THERMO_BASE_DIR") else None,
                    Path("/content"), Path("/mnt/data"), Path.cwd(), Path("/tmp")]
_candidate_bases = [p for p in _candidate_bases if p is not None]

def _base_is_usable(p):
    try:
        return p.exists() and os.access(p, os.W_OK)
    except Exception:
        return False

BASE_DIR = next((p for p in _candidate_bases if _base_is_usable(p)), Path("/tmp"))
PROJECT_NAME = os.environ.get("THERMO_PROJECT_NAME", "project_thermography_equine")
PROJECT_ROOT = BASE_DIR / PROJECT_NAME
DATA_ROOT = PROJECT_ROOT / "data"
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR = OUTPUT_ROOT / "figures"
MODELS_DIR = OUTPUT_ROOT / "models"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"
XAI_DIR = OUTPUT_ROOT / "xai_integrated_gradients"
for d in [PROJECT_ROOT, DATA_ROOT, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, GRADCAM_DIR, XAI_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SEARCH_ROOTS = [PROJECT_ROOT, OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR, GRADCAM_DIR, BASE_DIR, Path('/content'), Path('/mnt/data'), Path.cwd()]

def find_first(patterns, roots=SEARCH_ROOTS, required=False):
    """Find a file, respecting the order of patterns.

    The previous version pooled all patterns and then selected the newest file.
    That could load error_case_review.csv instead of cnn_model_predictions.csv.
    """
    for pat in patterns:
        hits=[]
        for root in roots:
            root=Path(root)
            if not root.exists():
                continue
            hits.extend([p for p in root.rglob(pat) if p.is_file() and '.ipynb_checkpoints' not in p.parts])
        hits=sorted(set(hits), key=lambda p: p.stat().st_mtime, reverse=True)
        if hits:
            return hits[0]
    if required:
        raise FileNotFoundError(f"Could not find any of: {patterns}")
    return None

def read_csv_found(patterns, required_cols=None, required=True):
    path=find_first(patterns, required=required)
    if path is None:
        return None, None
    df=pd.read_csv(path)
    if required_cols:
        missing=[c for c in required_cols if c not in df.columns]
        if missing:
            raise KeyError(f"{path} missing required columns: {missing}")
    print(f"Loaded {path} shape={df.shape}")
    return df, path

def save_text(path, text):
    path=Path(path); path.parent.mkdir(parents=True, exist_ok=True); path.write_text(text, encoding='utf-8')

print('BASE_DIR:', BASE_DIR)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)

# Integrated Gradients without Captum. This notebook uses the same ResNet18 checkpoint and image preprocessing
# convention as Notebook 11. It creates an additional XAI method for sensitivity analysis.
import zipfile
from PIL import Image
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    from torchvision import models, transforms
except Exception as e:
    raise ImportError("PyTorch and torchvision are required for Integrated Gradients analysis.") from e

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = int(os.environ.get("THERMO_CNN_SIZE", "224"))
N_STEPS = int(os.environ.get("THERMO_IG_STEPS", "64"))
TARGET_MODE = os.environ.get("THERMO_IG_TARGET_MODE", "positive_logit")
IMAGE_EXTS = {'.png','.jpg','.jpeg','.tif','.tiff','.bmp','.webp'}

# Locate checkpoint and predictions.
checkpoint_path = find_first(["cnn_resnet18_best.pt", "*resnet18*best*.pt", "*.pt"], required=True)
pred, pred_path = read_csv_found(["cnn_model_predictions.csv", "gradcam_summary*.csv", "error_case_review*.csv"],
    required_cols=["split", "label_binary", "image_name"], required=True)
if "cnn_probability_pathological" not in pred.columns and "gradcam_model_probability_recomputed" in pred.columns:
    pred["cnn_probability_pathological"] = pred["gradcam_model_probability_recomputed"]
if "cnn_predicted_label_binary" not in pred.columns:
    pred["cnn_predicted_label_binary"] = (pd.to_numeric(pred.get("cnn_probability_pathological", 0.5), errors='coerce').fillna(0.5) >= 0.5).astype(int)

# Prefer locked test set.
test = pred[pred['split'].astype(str).str.lower().eq('test')].copy()
if test.empty:
    test = pred.copy()

# Resolve image paths robustly. This handles:
# 1) absolute paths saved in older notebooks, 2) paths relative to PROJECT_ROOT/DATA_ROOT,
# 3) image_name values that are full paths, and 4) basename/stem-only matching.
# It also reports unresolved images with a concrete diagnostic.
PREFERRED_IMAGE_COLS = [
    'resolved_image_path_ig', 'resolved_image_path', 'clean_image_path', 'feature_image_path',
    'image_path', 'filepath', 'file_path', 'path', 'filename', 'file_name', 'image_name'
]

# You can force an image folder in Colab before running this cell, e.g.:
# %env THERMO_IMAGE_ROOT=/content/project_thermography_equine/data/raw_images
EXTRA_IMAGE_ROOTS = []
for key in ['THERMO_IMAGE_ROOT', 'THERMO_DATA_ROOT', 'THERMO_RAW_IMAGE_ROOT']:
    val = os.environ.get(key, '').strip()
    if val:
        EXTRA_IMAGE_ROOTS.append(Path(val))

IMAGE_SEARCH_ROOTS = []
for root in EXTRA_IMAGE_ROOTS + [
    PROJECT_ROOT, DATA_ROOT, PROJECT_ROOT/'data', PROJECT_ROOT/'data/raw', PROJECT_ROOT/'data/raw_images',
    PROJECT_ROOT/'data/images', PROJECT_ROOT/'data/test', PROJECT_ROOT/'data/train', PROJECT_ROOT/'data/valid',
    OUTPUT_ROOT, CONFIG_DIR, REPORTS_DIR, TABLES_DIR, FIGURES_DIR, GRADCAM_DIR,
    BASE_DIR, Path('/content'), Path('/content/drive/MyDrive'), Path('/mnt/data'), Path.cwd()
]:
    root = Path(root)
    if root.exists() and root not in IMAGE_SEARCH_ROOTS:
        IMAGE_SEARCH_ROOTS.append(root)

# Optional: unpack image archives if they were uploaded. The previous version only checked a shallow set
# of ZIP locations; this version searches recursively and supports common archive formats.
def _try_unpack_archive(archive_path):
    archive_path = Path(archive_path)
    out_dir = DATA_ROOT / ('unpacked_' + archive_path.stem.replace(' ', '_'))
    marker = out_dir / '.unpacked_ok'
    if marker.exists():
        return out_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    suffixes = ''.join(archive_path.suffixes).lower()
    try:
        if archive_path.suffix.lower() == '.zip':
            with zipfile.ZipFile(archive_path, 'r') as zf:
                zf.extractall(out_dir)
        elif suffixes.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2', '.tbz2')):
            import tarfile
            with tarfile.open(archive_path, 'r:*') as tf:
                tf.extractall(out_dir)
        elif archive_path.suffix.lower() in {'.rar', '.7z'}:
            # Colab usually has 7z available. If not, this will simply warn.
            import subprocess
            subprocess.run(['7z', 'x', str(archive_path), '-o' + str(out_dir), '-y'], check=True)
        else:
            return None
        marker.write_text('ok', encoding='utf-8')
        return out_dir
    except Exception as e:
        warnings.warn(f'Could not unpack {archive_path}: {e}')
        return None

archive_exts = {'.zip', '.rar', '.7z', '.tar', '.gz', '.tgz', '.bz2'}
archive_roots = [r for r in [PROJECT_ROOT, DATA_ROOT, BASE_DIR, Path('/content'), Path('/content/drive/MyDrive'), Path('/mnt/data')] if r.exists()]
archives = []
for root in archive_roots:
    try:
        archives.extend([p for p in root.rglob('*') if p.is_file() and (p.suffix.lower() in archive_exts or ''.join(p.suffixes).lower().endswith(('.tar.gz','.tar.bz2')))])
    except Exception as e:
        warnings.warn(f'Could not scan archives in {root}: {e}')
for ap in sorted(set(archives), key=lambda p: p.stat().st_mtime, reverse=True):
    unpacked = _try_unpack_archive(ap)
    if unpacked is not None and unpacked.exists() and unpacked not in IMAGE_SEARCH_ROOTS:
        IMAGE_SEARCH_ROOTS.insert(0, unpacked)

print('Image search roots:')
for r in IMAGE_SEARCH_ROOTS:
    print(' -', r)

# Build a one-time image index. This is faster and more reliable than rglob for every row.
image_paths = []
for root in IMAGE_SEARCH_ROOTS:
    try:
        image_paths.extend([p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS and '.ipynb_checkpoints' not in p.parts])
    except Exception as e:
        warnings.warn(f'Could not scan {root}: {e}')
image_paths = sorted(set(image_paths), key=lambda p: p.stat().st_mtime, reverse=True)

by_name = {}
by_stem = {}
for p in image_paths:
    by_name.setdefault(p.name.lower(), p)
    by_stem.setdefault(p.stem.lower(), p)

print(f'Indexed {len(image_paths)} image files for Integrated Gradients.')
if image_paths[:10]:
    print('Example indexed images:', [str(p) for p in image_paths[:10]])

# Extra diagnostics: show how many actual files exist in data-like folders.
scan_summary = []
for root in IMAGE_SEARCH_ROOTS:
    try:
        n_img = sum(1 for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)
        if n_img:
            scan_summary.append({'root': str(root), 'image_files': n_img})
    except Exception:
        pass
pd.DataFrame(scan_summary).to_csv(CONFIG_DIR / 'integrated_gradients_image_scan_summary.csv', index=False)

def _candidate_strings(row):
    vals = []
    for col in PREFERRED_IMAGE_COLS:
        if col in row.index and pd.notna(row[col]):
            s = str(row[col]).strip().strip('"').strip("'")
            if s and s.lower() not in {'nan', 'none'}:
                vals.append(s)
    out=[]
    for v in vals:
        if v not in out:
            out.append(v)
    return out

def _resolve_one_string(s):
    p = Path(s)
    if p.is_absolute() and p.exists() and p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        return p
    # try as-is and URL/Windows-normalized variants
    variants = [s, s.replace('\\', '/')]
    for v in list(variants):
        variants.append(Path(v).name)
    for v in variants:
        for root in IMAGE_SEARCH_ROOTS:
            q = root / v
            if q.exists() and q.is_file() and q.suffix.lower() in IMAGE_EXTS:
                return q
    fname = p.name.lower()
    if fname in by_name:
        return by_name[fname]
    stem = p.stem.lower()
    if stem in by_stem:
        return by_stem[stem]
    if stem:
        for img in image_paths:
            st = img.stem.lower()
            if stem == st or stem in st or st in stem:
                return img
    return None

def resolve_image_path(row):
    for s in _candidate_strings(row):
        hit = _resolve_one_string(s)
        if hit is not None:
            return str(hit)
    return None

test['resolved_image_path_ig'] = test.apply(resolve_image_path, axis=1)
unresolved = test[test['resolved_image_path_ig'].isna()].copy()
if not unresolved.empty:
    unresolved_path = CONFIG_DIR / 'integrated_gradients_unresolved_images.csv'
    unresolved.to_csv(unresolved_path, index=False)
    print(f'Warning: {len(unresolved)} rows still unresolved. Saved details to {unresolved_path}')
    print('First unresolved image_name values:', unresolved['image_name'].head(10).astype(str).tolist())

test = test[test['resolved_image_path_ig'].notna()].copy()
if test.empty:
    expected = sorted(set(unresolved['image_name'].dropna().astype(str).head(20).tolist())) if 'image_name' in unresolved.columns else []
    msg = (
        'No test JPG/PNG/TIF images could be resolved for Integrated Gradients.\n\n'
        f'The prediction table was loaded correctly from: {pred_path}\n'
        f'But the scanner found only {len(image_paths)} image file(s) under all search roots. '
        'Those files are probably ROC/diagnostic figures, not the original thermography photos.\n\n'
        'What is missing: the original image files referenced by cnn_model_predictions.csv, e.g.:\n'
        + '\n'.join([' - ' + x for x in expected[:20]]) + '\n\n'
        'Fix in Colab: upload/extract the folder or ZIP containing these JPG files into '
        '/content/project_thermography_equine/data, or set before running this notebook:\n'
        '  %env THERMO_IMAGE_ROOT=/full/path/to/folder/with/jpgs\n\n'
        f'Diagnostics saved to:\n - {CONFIG_DIR / "integrated_gradients_unresolved_images.csv"}\n'
        f' - {CONFIG_DIR / "integrated_gradients_image_scan_summary.csv"}'
    )
    raise FileNotFoundError(msg)
print(f'Resolved {len(test)} test images for Integrated Gradients.')

# Load model.
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
ckpt = torch.load(checkpoint_path, map_location=DEVICE)
state = ckpt.get('model_state_dict', ckpt)
model.load_state_dict(state)
model.to(DEVICE)
model.eval()

tfm = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
# Baseline in normalized tensor space: black RGB image after normalization.
baseline = torch.zeros((1,3,IMG_SIZE,IMG_SIZE), device=DEVICE)

mean = torch.tensor([0.485,0.456,0.406], device=DEVICE).view(1,3,1,1)
std = torch.tensor([0.229,0.224,0.225], device=DEVICE).view(1,3,1,1)

def integrated_gradients(x, predicted_label):
    # x: normalized tensor [1,3,H,W]
    scaled = [baseline + (float(i)/N_STEPS)*(x-baseline) for i in range(1, N_STEPS+1)]
    grads=[]
    for xi in scaled:
        xi = xi.clone().detach().requires_grad_(True)
        model.zero_grad(set_to_none=True)
        logit = model(xi).squeeze()
        if TARGET_MODE == 'predicted_class':
            target = logit if int(predicted_label)==1 else -logit
        elif TARGET_MODE == 'positive_logit':
            target = logit
        else:
            raise ValueError("THERMO_IG_TARGET_MODE must be positive_logit or predicted_class")
        target.backward()
        grads.append(xi.grad.detach())
    avg_grads = torch.stack(grads).mean(dim=0)
    attr = (x - baseline) * avg_grads
    # Aggregate absolute attribution over RGB channels and normalize to [0,1].
    heat = attr.detach().abs().sum(dim=1).squeeze().cpu().numpy()
    heat = heat - np.nanmin(heat)
    if np.nanmax(heat) > 0:
        heat = heat / np.nanmax(heat)
    return heat

rows=[]
for _, row in test.iterrows():
    image_path = Path(row['resolved_image_path_ig'])
    image_name = str(row['image_name'])
    out_prefix = XAI_DIR / f"test_{Path(image_name).stem}"
    try:
        img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
        x = tfm(img).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            prob = float(torch.sigmoid(model(x).squeeze()).detach().cpu())
        heat = integrated_gradients(x, row.get('cnn_predicted_label_binary', int(prob>=0.5)))
        heat_path = out_prefix.with_name(out_prefix.name + '_ig_heatmap.npy')
        overlay_path = out_prefix.with_name(out_prefix.name + '_ig_overlay.png')
        np.save(heat_path, heat)
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.imshow(heat, alpha=0.45)
        plt.axis('off')
        plt.tight_layout(pad=0)
        plt.savefig(overlay_path, dpi=200, bbox_inches='tight', pad_inches=0)
        plt.close()
        max_y, max_x = np.unravel_index(np.nanargmax(heat), heat.shape)
        rows.append({
            'horse_id': row.get('horse_id', np.nan),
            'image_name': image_name,
            'split': row.get('split', 'test'),
            'label_binary': int(row['label_binary']),
            'label_clinical': row.get('label_clinical', np.nan),
            'cnn_probability_pathological': row.get('cnn_probability_pathological', prob),
            'ig_model_probability_recomputed': prob,
            'ig_heatmap_path': str(heat_path),
            'ig_overlay_path': str(overlay_path),
            'ig_max_x': int(max_x),
            'ig_max_y': int(max_y),
            'ig_image_width': IMG_SIZE,
            'ig_image_height': IMG_SIZE,
            'ig_steps': N_STEPS,
            'ig_target_mode': TARGET_MODE,
            'ig_status': 'ok',
            'ig_error': None,
        })
    except Exception as e:
        rows.append({
            'horse_id': row.get('horse_id', np.nan), 'image_name': image_name, 'split': row.get('split','test'),
            'label_binary': int(row['label_binary']), 'ig_status': 'failed', 'ig_error': repr(e)
        })

ig_summary = pd.DataFrame(rows)
for p in [CONFIG_DIR/"integrated_gradients_summary.csv", REPORTS_DIR/"integrated_gradients_summary.csv", TABLES_DIR/"table_integrated_gradients_summary.csv"]:
    ig_summary.to_csv(p, index=False)

status={
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'analysis': 'integrated_gradients_xai',
    'checkpoint_path': str(checkpoint_path),
    'predictions_input': str(pred_path),
    'n_requested': int(len(test)),
    'n_generated': int((ig_summary['ig_status']=='ok').sum()),
    'ig_steps': N_STEPS,
    'target_mode': TARGET_MODE,
    'device': str(DEVICE),
}
save_text(CONFIG_DIR/"integrated_gradients_status.json", json.dumps(status, indent=2))
save_text(REPORTS_DIR/"methods_integrated_gradients_text.txt", "Integrated Gradients was used as an additional attribution-based XAI sensitivity analysis. The same locked test images, ResNet18 checkpoint, preprocessing, and positive-class target convention used for Grad-CAM were applied. Attribution maps were generated by integrating input gradients along a straight-line path from a black baseline image to the observed thermographic image, then aggregating absolute attributions across RGB channels and normalizing the resulting heatmap to [0,1].\n")
print(json.dumps(status, indent=2))
print(ig_summary.head().to_string(index=False))


BASE_DIR: /content
PROJECT_ROOT: /content/project_thermography_equine
OUTPUT_ROOT: /content/project_thermography_equine/outputs
Loaded /content/project_thermography_equine/outputs/config/cnn_model_predictions.csv shape=(347, 9)
Image search roots:
 - /content/project_thermography_equine/data/unpacked_clean_images_224x224
 - /content/project_thermography_equine
 - /content/project_thermography_equine/data
 - /content/project_thermography_equine/outputs
 - /content/project_thermography_equine/outputs/config
 - /content/project_thermography_equine/outputs/reports
 - /content/project_thermography_equine/outputs/tables
 - /content/project_thermography_equine/outputs/figures
 - /content/project_thermography_equine/outputs/gradcam
 - /content
Indexed 349 image files for Integrated Gradients.
Example indexed images: ['/content/project_thermography_equine/data/unpacked_clean_images_224x224/clean_images/train/pathological/4Y05M.jpg', '/content/project_thermography_equine/data/unpacked_clean_imag

/tmp/ipykernel_3767/1082128438.py:165: UserWarning: Could not unpack /content/dataset_split-20260605T090743Z-3-001.zip: File is not a zip file
  warnings.warn(f'Could not unpack {archive_path}: {e}')


Resolved 53 test images for Integrated Gradients.
{
  "created_utc": "2026-06-07T11:36:06.984723+00:00",
  "analysis": "integrated_gradients_xai",
  "checkpoint_path": "/content/project_thermography_equine/outputs/config/cnn_resnet18_best.pt",
  "predictions_input": "/content/project_thermography_equine/outputs/config/cnn_model_predictions.csv",
  "n_requested": 53,
  "n_generated": 53,
  "ig_steps": 64,
  "target_mode": "positive_logit",
  "device": "cpu"
}
horse_id image_name split  label_binary label_clinical  cnn_probability_pathological  ig_model_probability_recomputed                                                                                 ig_heatmap_path                                                                                 ig_overlay_path  ig_max_x  ig_max_y  ig_image_width  ig_image_height  ig_steps ig_target_mode ig_status ig_error
   0A0F5  0A0F5.jpg  test             0        healthy                      0.001057                         0.002458 /content/pro

## Completion
The notebook writes attribution maps, overlays, summary tables, and methods text. Localization metrics for these maps can be evaluated by adapting notebook 12 to use `integrated_gradients_summary.csv`.